# MIMIC-III NLP Tutorial: spaCy vs scispaCy vs medspaCy
### Cohort: Subarachnoid Hemorrhage (ICD-9 430) — Discharge Summaries



## Setup & Installs

In [ ]:
# Remove preinstalled versions
!pip uninstall -y spacy scispacy medspacy pyrush quickumls

# Install compatible versions
!pip install -q spacy==3.7.5
!pip install -q scispacy==0.5.4
!pip install -q medspacy==1.2.0

# Install scispaCy models
!pip install -q \
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_core_sci_sm-0.5.4.tar.gz

!pip install -q \
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bc5cdr_md-0.5.4.tar.gz

# General English model
!python -m spacy download en_core_web_sm

import spacy
import scispacy
import medspacy

print("spaCy:", spacy.__version__)
print("scispaCy:", scispacy.__version__)
print("medspaCy:", medspacy.__version__)

Found existing installation: spacy 3.7.5
Uninstalling spacy-3.7.5:
  Successfully uninstalled spacy-3.7.5
Found existing installation: PyRuSH 1.0.13
Uninstalling PyRuSH-1.0.13:
  Successfully uninstalled PyRuSH-1.0.13


In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

import spacy
import scispacy
import medspacy
from medspacy.visualization import visualize_ent

from gensim.models import Word2Vec
from sklearn.manifold import TSNE

pd.set_option('display.max_colwidth', 200)
np.random.seed(42)


ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## Cohort Extraction (BigQuery)

Pull **discharge summaries** for patients diagnosed with **ICD-9 code 430**
(subarachnoid hemorrhage),


In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

project_id = 'mc-ut-msai-aih-1'
client = bigquery.Client(project=project_id)
print("BigQuery client connected!")


In [ ]:
icd9_430_query = """
SELECT
    n.row_id,
    n.subject_id,
    n.hadm_id,
    n.chartdate,
    n.category,
    n.description,
    n.text
FROM `physionet-data.mimiciii_notes.noteevents` n
JOIN (
    SELECT DISTINCT subject_id
    FROM `physionet-data.mimiciii_clinical.diagnoses_icd`
    WHERE icd9_code = '430'
) sah
ON n.subject_id = sah.subject_id
WHERE n.category = 'Discharge summary'
"""

df_notes = client.query(icd9_430_query).to_dataframe()
print(df_notes.shape)
df_notes.head()


In [ ]:
# Basic sanity checks
print("Number of notes:", len(df_notes))
print("Number of unique patients:", df_notes['subject_id'].nunique())
df_notes['text_len'] = df_notes['text'].str.len()
df_notes['text_len'].describe()


## Preprocessing

MIMIC notes contain de-identification placeholders like `[**2138-8-20**]` or
`[**Hospital 4**]`. We strip these out and do light cleanup before running NLP.


In [ ]:
def clean_note(text):
    text = re.sub(r'\[\*\*.*?\*\*\]', '', text)   # remove de-id placeholders
    text = re.sub(r'\s+', ' ', text)                  # collapse whitespace
    return text.strip()

df_notes['clean_text'] = df_notes['text'].apply(clean_note)

# keep 300 sample size
SAMPLE_SIZE = min(300, len(df_notes))
df_sample = df_notes.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Using {len(df_sample)} notes for this tutorial")
df_sample[['subject_id', 'clean_text']].head(2)


---
## 1. spaCy

General-purpose NLP


In [ ]:
nlp_spacy = spacy.load("en_core_web_sm")

docs_spacy = list(nlp_spacy.pipe(df_sample['clean_text'].tolist()))

# Look at entities in the first note
for ent in docs_spacy[0].ents:
    print(ent.text, "->", ent.label_)


### 1a. (Optional) displaCy visualization of entities in one note

In [ ]:
from spacy import displacy

displacy.render(docs_spacy[0], style="ent", jupyter=True)


### 1b. Entity extraction across the cohort

In [ ]:
spacy_entities = []
for doc in docs_spacy:
    for ent in doc.ents:
        spacy_entities.append({'text': ent.text, 'label': ent.label_})

df_spacy_ents = pd.DataFrame(spacy_entities)
print("Total entities extracted:", len(df_spacy_ents))
df_spacy_ents['label'].value_counts()


### 1c. word2vec on the note corpus

In [ ]:
# Tokenize each note into a list of lowercase alphabetic tokens
def tokenize(doc):
    return [t.text.lower() for t in doc if t.is_alpha and not t.is_stop]

spacy_sentences = [tokenize(doc) for doc in docs_spacy]

w2v_spacy = Word2Vec(
    sentences=spacy_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    seed=42,
)
print("Vocab size:", len(w2v_spacy.wv))


### 1d. Tuned t-SNE

We compare several perplexity values (per the Distill.pub guidance) before settling on one.

In [ ]:
def plot_tsne_perplexities(word2vec_model, perplexities, title_prefix, max_words=300):
    words = list(word2vec_model.wv.index_to_key)[:max_words]
    vectors = np.array([word2vec_model.wv[w] for w in words])

    fig, axes = plt.subplots(1, len(perplexities), figsize=(6 * len(perplexities), 6))
    if len(perplexities) == 1:
        axes = [axes]

    for ax, perp in zip(axes, perplexities):
        tsne = TSNE(
            n_components=2,
            perplexity=min(perp, max(5, len(words) - 1)),
            n_iter=2000,
            init='pca',
            random_state=42,
        )
        coords = tsne.fit_transform(vectors)
        ax.scatter(coords[:, 0], coords[:, 1], s=10, alpha=0.6)
        ax.set_title(f"{title_prefix} (perplexity={perp})")

    plt.tight_layout()
    plt.show()

plot_tsne_perplexities(w2v_spacy, perplexities=[5, 15, 30, 50], title_prefix="spaCy word2vec")


---
## 2. scispaCy

Biomedical-domain NLP


In [ ]:
nlp_scispacy = spacy.load("en_core_sci_sm")

docs_scispacy = list(nlp_scispacy.pipe(df_sample['clean_text'].tolist()))

for ent in docs_scispacy[0].ents:
    print(ent.text, "->", ent.label_)


### 2a. Entity extraction across the cohort

In [ ]:
scispacy_entities = []
for doc in docs_scispacy:
    for ent in doc.ents:
        scispacy_entities.append({'text': ent.text, 'label': ent.label_})

df_scispacy_ents = pd.DataFrame(scispacy_entities)
print("Total entities extracted:", len(df_scispacy_ents))
df_scispacy_ents['label'].value_counts()


### 2b. word2vec on the note corpus

In [ ]:
scispacy_sentences = [tokenize(doc) for doc in docs_scispacy]

w2v_scispacy = Word2Vec(
    sentences=scispacy_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    seed=42,
)
print("Vocab size:", len(w2v_scispacy.wv))


### 2c. Tuned t-SNE

In [ ]:
plot_tsne_perplexities(w2v_scispacy, perplexities=[5, 15, 30, 50], title_prefix="scispaCy word2vec")


3---
## 6. medspaCy

Clinical NLP pipeline


In [ ]:
nlp_medspacy = medspacy.load()

docs_medspacy = list(nlp_medspacy.pipe(df_sample['clean_text'].tolist()))

for ent in docs_medspacy[0].ents:
    print(ent.text, "-> negated:", ent._.is_negated, "| family:", ent._.is_family, "| hypothetical:", ent._.is_hypothetical)


### 3a. Visualize context attributes (negation, family history, etc.)

In [ ]:
visualize_ent(docs_medspacy[0])


### 3b. Entity extraction across the cohort (with context attributes)

In [ ]:
medspacy_entities = []
for doc in docs_medspacy:
    for ent in doc.ents:
        medspacy_entities.append({
            'text': ent.text,
            'label': ent.label_,
            'is_negated': ent._.is_negated,
            'is_family': ent._.is_family,
            'is_hypothetical': ent._.is_hypothetical,
        })

df_medspacy_ents = pd.DataFrame(medspacy_entities)
print("Total entities extracted:", len(df_medspacy_ents))
print("\nNegated entities:", df_medspacy_ents['is_negated'].sum())
df_medspacy_ents['label'].value_counts()


### 3c. word2vec on the note corpus

In [ ]:
medspacy_sentences = [tokenize(doc) for doc in docs_medspacy]

w2v_medspacy = Word2Vec(
    sentences=medspacy_sentences,
    vector_size=100,
    window=5,
    min_count=3,
    workers=4,
    seed=42,
)
print("Vocab size:", len(w2v_medspacy.wv))


### 3d. Tuned t-SNE

In [ ]:
plot_tsne_perplexities(w2v_medspacy, perplexities=[5, 15, 30, 50], title_prefix="medspaCy word2vec")


---
## 4. Compare & Contrast

Summary table of entity counts and unique labels found by each tool.


In [ ]:
comparison = pd.DataFrame({
    'Tool': ['spaCy', 'scispaCy', 'medspaCy'],
    'Total Entities': [len(df_spacy_ents), len(df_scispacy_ents), len(df_medspacy_ents)],
    'Unique Labels': [df_spacy_ents['label'].nunique(), df_scispacy_ents['label'].nunique(), df_medspacy_ents['label'].nunique()],
})
comparison


---
## 9. Conclusion

*(Fill in after running the notebook)* Summarize which tool performed best for
which purpose, and what you'd recommend for downstream ML feature engineering
using MIMIC notes.
